# Chapter 4: Shifting, Lags, and Differences

## Learning Objectives

In this chapter, we learn how to retrieve previous observations, calculate changes over time, and create time-based features using Pandas.

## Topics Covered

- Shifting observations with `shift()`
- Understanding positive and negative shifts
- Creating lag features such as `lag_1` and `lag_2`
- Calculating absolute changes using `diff()`
- Calculating changes over multiple periods using `diff(2)`
- Calculating relative changes using `pct_change()`
- Combining lag, difference, and percentage-change features
- Applying these techniques to time-series machine learning

## Why This Matters

Time-series analysis often involves understanding how the past relates to the present and future.

In chemical engineering, historical reactor temperatures, pressures, production rates, and other process measurements can help us understand process behaviour and build forecasting models.

Lag features provide historical information, while differences and percentage changes describe how measurements evolve over time.

These techniques are useful for exploratory data analysis, process monitoring, anomaly detection, and machine-learning feature engineering.

**Important:** A lag represents a previous observation, not necessarily a fixed duration. If observations are irregularly spaced, `shift(1)` refers to the previous row rather than exactly one hour earlier.

In [1]:
import pandas as pd

dates = pd.date_range(
    start="2026-01-01 09:00",
    periods=5,
    freq="h"
)

df = pd.DataFrame(
    {"temperature": [100, 104, 109, 107, 115]},
    index=dates
)

df.index.name = "time"

df

,temperature
time,
2026-01-01 09:00:00,100
2026-01-01 10:00:00,104
2026-01-01 11:00:00,109
2026-01-01 12:00:00,107
2026-01-01 13:00:00,115


In [2]:
# Using shift
df["previous_temperature"] = df["temperature"].shift(1)

df

,temperature,previous_temperature
time,,
2026-01-01 09:00:00,100,NaN
2026-01-01 10:00:00,104,100.0
2026-01-01 11:00:00,109,104.0
2026-01-01 12:00:00,107,109.0
2026-01-01 13:00:00,115,107.0


shift(1) moves the values down by one row. Each row now contains the temperature from the previous observation.
- At 10:00, the previous temperature was 100°C.
- At 11:00, it was 104°C.
- At 09:00, no earlier observation exists, so Pandas inserts NaN.
Notice that shift(1) moves the data values, not the timestamps in the index. That distinction matters.

In [3]:
# Now each row contains the temperature from two observations earlier.


df["temperature_2_periods_ago"] = df["temperature"].shift(2)

df

,temperature,previous_temperature,temperature_2_periods_ago
time,,,
2026-01-01 09:00:00,100,NaN,NaN
2026-01-01 10:00:00,104,100.0,NaN
2026-01-01 11:00:00,109,104.0,100.0
2026-01-01 12:00:00,107,109.0,104.0
2026-01-01 13:00:00,115,107.0,109.0


A lag is a past observation used alongside the current observation.
 - df["lag_1"] = df["temperature"].shift(1)
 - df["lag_2"] = df["temperature"].shift(2)
 - lag_1: temperature one observation ago.
 - lag_2: temperature two observations ago.

In machine learning, these can help predict future reactor temperatures using historical measurements.
One important rule: when forecasting, make sure your features contain only information that would actually be available at prediction time. Otherwise, your model may look brilliant in testing and then fall flat in reality, which is an annoyingly common trick of bad modeling.


In [4]:
# .diff()
df["temperature_change"] = df["temperature"].diff()

df

,temperature,previous_temperature,temperature_2_periods_ago,temperature_change
time,,,,
2026-01-01 09:00:00,100,NaN,NaN,NaN
2026-01-01 10:00:00,104,100.0,NaN,4.0
2026-01-01 11:00:00,109,104.0,100.0,5.0
2026-01-01 12:00:00,107,109.0,104.0,-2.0
2026-01-01 13:00:00,115,107.0,109.0,8.0


- shift(2) moves the values by two rows, letting you see the value from two observations earlier.
- diff(2) calculates the difference between the current value and the value from two observations earlier.

In [5]:
# Percentage change
df["temperature_pct_change"] = (
    df["temperature"].pct_change() * 100  #pct_change() returns a fraction by default so we multiply by 100 to display percentages.
)

df

,temperature,previous_temperature,temperature_2_periods_ago,temperature_change,temperature_pct_change
time,,,,,
2026-01-01 09:00:00,100,NaN,NaN,NaN,NaN
2026-01-01 10:00:00,104,100.0,NaN,4.0,4.000000
2026-01-01 11:00:00,109,104.0,100.0,5.0,4.807692
2026-01-01 12:00:00,107,109.0,104.0,-2.0,-1.834862
2026-01-01 13:00:00,115,107.0,109.0,8.0,7.476636


In [6]:
production = pd.Series([200, 230, 220, 242])

features = pd.DataFrame({
    "production": production,
    "lag_1": production.shift(1),
    "change": production.diff(),
    "pct_change": production.pct_change() * 100
})

features

,production,lag_1,change,pct_change
0,200,NaN,NaN,NaN
1,230,200.0,30.0,15.000000
2,220,230.0,-10.0,-4.347826
3,242,220.0,22.0,10.000000


# Chapter 4: Summary and Knowledge Check

## 1. Summary

### Shifting with `shift()`

- `series.shift(1)` moves values down by one row.
- `series.shift(-1)` moves values up by one row.
- `series.shift(2)` moves values down by two rows.
- Missing values (`NaN`) appear where no corresponding observation exists.
- By default, `shift()` moves the data values without shifting the index labels.

### Lag Features

Lag features represent historical observations.

```python
df["lag_1"] = df["temperature"].shift(1)
df["lag_2"] = df["temperature"].shift(2)
```

These features can help machine-learning models learn relationships between previous and future measurements.

### Absolute Differences with `diff()`

The default difference is:

\[
\text{diff}_t = X_t - X_{t-1}
\]

```python
df["temperature_change"] = df["temperature"].diff()
```

- A positive difference indicates an increase.
- A negative difference indicates a decrease.
- A zero difference indicates no change.
- The first difference is `NaN` because no previous observation exists.

`diff(2)` calculates the difference between the current observation and the observation two rows earlier.

### Percentage Changes with `pct_change()`

Percentage change measures the relative change from the previous observation.

\[
\text{Percentage Change}_t =
\frac{X_t-X_{t-1}}{X_{t-1}}\times100
\]

```python
df["percentage_change"] = df["temperature"].pct_change() * 100
```

Pandas returns fractional changes by default, so multiplying by 100 expresses the result as a percentage.

Percentage changes can help compare relative changes across variables with different scales. However, they can be misleading when the previous value is zero or close to zero.

### Combining Time-Series Features

```python
df["lag_1"] = df["temperature"].shift(1)
df["absolute_change"] = df["temperature"].diff()
df["percentage_change"] = df["temperature"].pct_change() * 100
```

Each feature provides different information:

- `lag_1`: the previous measurement
- `absolute_change`: the absolute change from the previous measurement
- `percentage_change`: the relative change from the previous measurement

### Important Considerations

- `shift(1)` refers to the previous row, not necessarily the previous hour or day.
- The first lag and difference values are generally missing.
- Percentage changes require care when previous values are zero or very small.
- In forecasting, features must not use information unavailable at prediction time.
- Lag features can improve predictions, but their usefulness depends on the dataset and the forecasting task.

## 2. Knowledge Check

1. What does `shift(1)` do to a Series?
2. What is the difference between `shift(2)` and `diff(2)`?
3. Why does `diff()` return `NaN` for the first observation?
4. How does `pct_change()` differ from `diff()`?
5. A measurement increases from 80 to 100. What is its absolute change and percentage change?
6. Why can lag features be useful in forecasting?
7. Why does `shift(1)` not always mean one hour earlier?
8. What is data leakage, and how can it affect forecasting models?
9. Why should percentage changes be handled carefully when the previous value is zero?

## 3. Key Takeaway

`shift()` retrieves observations from other rows, `diff()` calculates absolute changes, and `pct_change()` calculates relative changes. Together, these methods help describe temporal patterns and construct useful features for time-series analysis and machine learning.